# Notebook 05 — Analysis

**No training here.** Loads existing checkpoints and produces the README's analysis figures:

1. **FVE distribution** across the 500 test activations — not just the mean.
2. **Best / worst reconstructions** — examples of where the pipeline succeeds and fails.
3. **Snippet length vs. FVE** — does context length matter?
4. **L2-norm bimodality** revisited from notebook 01.

In [ ]:
%pip install -q transformers==4.46.0 accelerate matplotlib "sympy>=1.13.3"
import os
MARKER = '/tmp/.kth_nla_kernel_restarted'
if not os.path.exists(MARKER):
    open(MARKER, 'w').close()
    print('Restarting kernel...')
    os.kill(os.getpid(), 9)
else:
    print('Packages installed; kernel already restarted.')

In [ ]:
import json, random
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/kth-nla-2026')
except ImportError:
    DATA_ROOT = Path('.')
print(f'data root: {DATA_ROOT}')

In [ ]:
MODEL_NAME = 'EleutherAI/pythia-160m'
LAYER = 6
DESC_TOKENS = 16
TRAIN_N, VAL_N, TEST_N = 4000, 500, 500
FIG_DIR = DATA_ROOT / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load everything
acts = torch.load(DATA_ROOT / 'data/activations/pythia160m_layer6_wikitext2_5000.pt', map_location='cpu')
with open(DATA_ROOT / 'data/activations/pythia160m_layer6_wikitext2_5000.jsonl') as f:
    records = [json.loads(line) for line in f]
with open(DATA_ROOT / 'data/av_descriptions.json') as f:
    av_descs = json.load(f)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token

perm = torch.randperm(acts.shape[0], generator=torch.Generator().manual_seed(SEED)).tolist()
test_idx = perm[TRAIN_N + VAL_N:TRAIN_N + VAL_N + TEST_N]
print(f'loaded {len(acts)} activations, {len(av_descs)} AV descriptions, {len(test_idx)} test indices')

## Reconstructor (frozen + retrained head from notebook 04)

In [ ]:
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device).eval()
for p in base.parameters(): p.requires_grad_(False)
HIDDEN = base.config.hidden_size

class Reconstructor(nn.Module):
    def __init__(self, base_model, layer, hidden):
        super().__init__()
        self.base = base_model
        self.captured = {}
        base_model.gpt_neox.layers[layer].register_forward_hook(self._hook)
        self.head = nn.Linear(hidden, hidden)
    def _hook(self, m, i, o):
        self.captured['h'] = o[0] if isinstance(o, tuple) else o
    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            _ = self.base(input_ids=input_ids, attention_mask=attention_mask)
        h_seq = self.captured['h']
        last_idx = attention_mask.sum(dim=1) - 1
        batch_idx = torch.arange(h_seq.size(0), device=h_seq.device)
        return self.head(h_seq[batch_idx, last_idx])

ar = Reconstructor(base, LAYER, HIDDEN).to(device).eval()
ar.head.load_state_dict(torch.load(DATA_ROOT / 'checkpoints/reconstructor_head_av_retrain.pt',
                                    map_location=device)['state_dict'])
for p in ar.parameters(): p.requires_grad_(False)
print('AR_retrain head loaded.')

## Per-test-sample FVE

Compute reconstruction quality for **each** test activation individually rather than averaging — lets us look at the distribution and pick out best/worst cases.

In [ ]:
@torch.no_grad()
def reconstruct_all():
    preds, trues, descs = [], [], []
    BATCH = 32
    for start in range(0, len(test_idx), BATCH):
        ids = test_idx[start:start + BATCH]
        z = [av_descs[i] for i in ids]
        h = torch.stack([acts[i] for i in ids]).to(device)
        enc = tok(z, padding=True, truncation=True, max_length=DESC_TOKENS,
                  return_tensors='pt').to(device)
        h_hat = ar(enc.input_ids, enc.attention_mask).cpu()
        preds.append(h_hat); trues.append(h.cpu()); descs.extend(z)
    return torch.cat(preds), torch.cat(trues), descs

h_pred, h_true, descs = reconstruct_all()
mean_act = h_true.mean(0)
var_per_sample = ((h_true - mean_act) ** 2).sum(dim=1)
err_per_sample = ((h_pred - h_true) ** 2).sum(dim=1)
fve_per_sample = 1.0 - err_per_sample / var_per_sample.mean()
cos_per_sample = F.cosine_similarity(h_pred, h_true, dim=1)
print(f'test set: {len(fve_per_sample)} samples')
print(f'mean FVE = {fve_per_sample.mean().item():+.4f}, median FVE = {fve_per_sample.median().item():+.4f}')
print(f'mean cos = {cos_per_sample.mean().item():+.4f}')

## Figure A — FVE distribution across test set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(fve_per_sample.numpy(), bins=40, color='tab:purple', alpha=0.8)
axes[0].axvline(0, color='k', lw=0.6, ls='--', label='mean-baseline (FVE=0)')
axes[0].axvline(fve_per_sample.mean(), color='red', lw=1, label=f'mean = {fve_per_sample.mean():.3f}')
axes[0].set_xlabel('per-sample FVE'); axes[0].set_ylabel('count')
axes[0].set_title('FVE distribution on the 500-sample test set')
axes[0].legend()

axes[1].scatter(cos_per_sample.numpy(), fve_per_sample.numpy(), s=8, alpha=0.4)
axes[1].axhline(0, color='k', lw=0.6, ls='--')
axes[1].set_xlabel('cosine similarity'); axes[1].set_ylabel('FVE')
axes[1].set_title('cosine vs FVE (per sample)')
plt.tight_layout()
plt.savefig(FIG_DIR / '05_fve_distribution.png', dpi=120)
plt.show()

## Figure B — best / worst reconstructions

Pick the 5 highest-FVE and 5 lowest-FVE test samples. Show the snippet, the AV description, and the FVE.

In [ ]:
order = torch.argsort(fve_per_sample, descending=True).tolist()
top5 = order[:5]; bot5 = order[-5:]

def show(label, sample_positions):
    print(f'=== {label} ===')
    for pos in sample_positions:
        global_idx = test_idx[pos]
        snippet = records[global_idx]['snippet'][:90].replace('\n', ' ')
        desc    = descs[pos]
        f       = fve_per_sample[pos].item()
        c       = cos_per_sample[pos].item()
        print(f'  FVE={f:+.3f}  cos={c:+.3f}')
        print(f'     snippet : {snippet!r}')
        print(f'     AV desc : {desc!r}')
    print()

show('TOP 5 (best reconstructions)',    top5)
show('BOTTOM 5 (worst reconstructions)', bot5)

## Figure C — snippet length vs. FVE

In [ ]:
snippet_lens = torch.tensor([records[i]['n_tokens'] for i in test_idx], dtype=torch.float32)
norm_per_sample = h_true.norm(dim=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].scatter(snippet_lens.numpy(), fve_per_sample.numpy(), s=8, alpha=0.4, color='tab:blue')
axes[0].axhline(0, color='k', lw=0.6, ls='--')
axes[0].set_xlabel('source snippet length (tokens)'); axes[0].set_ylabel('per-sample FVE')
axes[0].set_title('snippet length vs FVE')

axes[1].scatter(norm_per_sample.numpy(), fve_per_sample.numpy(), s=8, alpha=0.4, color='tab:orange')
axes[1].axhline(0, color='k', lw=0.6, ls='--')
axes[1].set_xlabel('||h|| (L2 norm of activation)'); axes[1].set_ylabel('per-sample FVE')
axes[1].set_title('activation magnitude vs FVE  (the bimodality revisited)')
plt.tight_layout()
plt.savefig(FIG_DIR / '05_fve_vs_length_and_norm.png', dpi=120)
plt.show()

## Numbers to lift into the README

In [ ]:
summary = {
    'test_n': len(fve_per_sample),
    'fve_mean':   float(fve_per_sample.mean()),
    'fve_median': float(fve_per_sample.median()),
    'fve_p10':    float(fve_per_sample.quantile(0.1)),
    'fve_p90':    float(fve_per_sample.quantile(0.9)),
    'cos_mean':   float(cos_per_sample.mean()),
    'cos_median': float(cos_per_sample.median()),
    'frac_positive_fve': float((fve_per_sample > 0).float().mean()),
}
for k, v in summary.items():
    print(f'  {k:20} {v:+.4f}')

with open(DATA_ROOT / 'analysis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('saved → analysis_summary.json')